# ai lab 6

name: ibrahim ahmed

roll number: 3093

### task 1: designing a heuristic for a warehouse robot

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

locations = {
    "receiving_area": (0, 0),
    "storage_a": (2, 1),
    "storage_b": (1, 4),
    "sorting_area": (4, 2),
    "inspection_area": (5, 5),
    "packing_station": (7, 6)
}

warehouse_graph = {
    "receiving_area": {"storage_a": 2.2, "storage_b": 4.1},
    "storage_a": {"sorting_area": 2.2},
    "storage_b": {"inspection_area": 5.0, "sorting_area": 6.0},
    "sorting_area": {"inspection_area": 3.2, "packing_station": 5.0},
    "inspection_area": {"packing_station": 2.2},
    "packing_station": {}
}

def heuristic(node, goal):
    x1, y1 = locations[node]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

goal = "packing_station"

print("heuristic values to packing_station")
for node in locations:
    print(node, ":", round(heuristic(node, goal), 2))

#### graph with the minimum cost path

In [ ]:
g = nx.DiGraph()
for node in warehouse_graph:
    for neighbor in warehouse_graph[node]:
        g.add_edge(node, neighbor, weight=warehouse_graph[node][neighbor])

path = nx.shortest_path(g, "receiving_area", "packing_station", weight="weight")
cost = nx.shortest_path_length(g, "receiving_area", "packing_station", weight="weight")

print("minimum cost path:", " -> ".join(path))
print("total cost:", round(cost, 2))

path_edges = []
for i in range(len(path) - 1):
    path_edges.append((path[i], path[i + 1]))

colors = []
for node in g.nodes():
    if node == "receiving_area":
        colors.append("red")
    elif node == "packing_station":
        colors.append("green")
    else:
        colors.append("lightblue")

plt.figure(figsize=(12, 7))
nx.draw(g, locations, with_labels=True, node_color=colors, node_size=1800, font_size=8, edge_color="gray")
nx.draw_networkx_edges(g, locations, edgelist=path_edges, edge_color="red", width=3)
nx.draw_networkx_edge_labels(g, locations, edge_labels=nx.get_edge_attributes(g, "weight"), font_size=7)
plt.title("warehouse robot (red = minimum cost path)")
plt.show()

#### checking the heuristic condition h(n) <= cost(n, m) + h(m)

In [ ]:
all_ok = True
for node in warehouse_graph:
    for neighbor in warehouse_graph[node]:
        hn = heuristic(node, goal)
        hm = heuristic(neighbor, goal)
        right = warehouse_graph[node][neighbor] + hm
        if hn <= right + 0.000001:
            status = "satisfied"
        else:
            status = "violated"
            all_ok = False
        print(node, "->", neighbor, "| h(n) =", round(hn, 2), "| cost + h(m) =", round(right, 2), "|", status)

if all_ok:
    print("all edges satisfy the condition")
else:
    print("some edges are slightly violated because edge costs are rounded")

**observation:** euclidean distance is a good heuristic here because the straight line distance is never more than the real path cost through the corridors, so it never overestimates the cost to reach the packing station. the condition h(n) <= cost(n, m) + h(m) holds on the edges (or is only slightly off where the edge costs are rounded).

### task 2: greedy best first search for airport baggage handling

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

locations = {
    "baggage_area": (0, 0),
    "security": (2, 1),
    "checkpoint": (1, 4),
    "food_court": (4, 2),
    "terminal_hall": (5, 5),
    "departure_gate": (8, 6)
}

airport_graph = {
    "baggage_area": {"security": 2.2, "checkpoint": 4.1},
    "security": {"food_court": 2.2},
    "checkpoint": {"terminal_hall": 5.0},
    "food_court": {"terminal_hall": 3.2, "departure_gate": 6.0},
    "terminal_hall": {"departure_gate": 3.2},
    "departure_gate": {}
}

def heuristic(node, goal):
    x1, y1 = locations[node]
    x2, y2 = locations[goal]
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def gbfs(start, goal):
    frontier = [start]
    parent = {start: None}
    visited = []

    while len(frontier) > 0:
        best = frontier[0]
        for node in frontier:
            if heuristic(node, goal) < heuristic(best, goal):
                best = node
        frontier.remove(best)
        visited.append(best)

        if best == goal:
            break

        for neighbor in airport_graph[best]:
            if neighbor not in visited and neighbor not in frontier:
                frontier.append(neighbor)
                parent[neighbor] = best

    path = []
    node = goal
    while node is not None:
        path.append(node)
        node = parent[node]
    path.reverse()

    cost = 0
    for i in range(len(path) - 1):
        cost = cost + airport_graph[path[i]][path[i + 1]]

    return path, cost, visited

start = "baggage_area"
goal = "departure_gate"

path, cost, visited = gbfs(start, goal)

print("expansion order:", " -> ".join(visited))
print("solution path:", " -> ".join(path))
print("total path cost:", round(cost, 2))
print()
print("gbfs only looks at h(n), the straight line distance to departure_gate.")
print("it picks checkpoint over security because its h value is smaller, even though it costs more.")
print("so gbfs finds a path fast but it is not always the cheapest one.")

#### graph with the gbfs path

In [ ]:
g = nx.DiGraph()
for node in airport_graph:
    for neighbor in airport_graph[node]:
        g.add_edge(node, neighbor, weight=airport_graph[node][neighbor])

path_edges = []
for i in range(len(path) - 1):
    path_edges.append((path[i], path[i + 1]))

colors = []
for node in g.nodes():
    if node == start:
        colors.append("red")
    elif node == goal:
        colors.append("green")
    elif node in visited:
        colors.append("yellow")
    else:
        colors.append("lightblue")

plt.figure(figsize=(12, 7))
nx.draw(g, locations, with_labels=True, node_color=colors, node_size=1800, font_size=8, edge_color="gray")
nx.draw_networkx_edges(g, locations, edgelist=path_edges, edge_color="red", width=3)
nx.draw_networkx_edge_labels(g, locations, edge_labels=nx.get_edge_attributes(g, "weight"), font_size=7)
plt.title("airport baggage handling - gbfs (red = solution path)")
plt.show()